# ThePainter — Evaluation & Promotion

Compares the current Production version of **ThePainter** (surrogate renderer) against
the latest training run (or a specific run). Visual comparison shows predicted vs
actual mel spectrograms for a set of validation specs.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
MLFLOW_URI   = "http://192.168.1.254:5000"
MODEL_NAME   = "ThePainter"
RUN_ID       = None   # set to override 'use latest run'
N_VAL        = 500    # validation samples
N_VIS        = 6      # number of spectrogram pairs to display


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import torch
import mlflow
from mlflow.tracking import MlflowClient
from rustic_ml.training.setup import setup_mlflow, setup_device
from rustic_ml.the_painter.inference import evaluate
from rustic_ml.mlflow_ui import show_register_widget, show_describe_widget

setup_mlflow(MLFLOW_URI, MODEL_NAME)
device = setup_device()
client = MlflowClient(MLFLOW_URI)
print(f"Device: {device}")

# TODO: build val_loader from ARDataset once ThePainter dataset is defined
# from rustic_ml.autoregressive.dataset import ARDataset
# val_ds = ARDataset(n_samples=N_VAL, vocab=vocab)
# val_loader = DataLoader(val_ds, batch_size=32, collate_fn=ar_collate_fn)
val_loader = None  # placeholder


In [ ]:
# ── Load production model ─────────────────────────────────────────────────
prod_model = None
prod_metrics = {}

prod_versions = client.get_latest_versions(MODEL_NAME, stages=["Production"])
if prod_versions:
    prod_model = mlflow.pytorch.load_model(prod_versions[0].source, map_location=device)
    prod_model.eval()
    print(f"Production: version {prod_versions[0].version} ({prod_versions[0].run_id[:8]})")
    prod_metrics = evaluate(prod_model, val_loader, device)
else:
    print("No Production version registered yet.")


In [ ]:
# ── Load candidate model ──────────────────────────────────────────────────
if RUN_ID is None:
    exp = client.get_experiment_by_name(MODEL_NAME)
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=["start_time DESC"],
        max_results=1,
    )
    RUN_ID = runs[0].info.run_id

cand_model = mlflow.pytorch.load_model(f"runs:/{RUN_ID}/model", map_location=device)
cand_model.eval()
print(f"Candidate run: {RUN_ID}")
cand_metrics = evaluate(cand_model, val_loader, device)


In [ ]:
# ── Metrics table ─────────────────────────────────────────────────────────
import pandas as pd

rows = []
for key in set(list(prod_metrics) + list(cand_metrics)):
    rows.append({
        "metric":     key,
        "production": prod_metrics.get(key, "—"),
        "candidate":  cand_metrics.get(key, "—"),
    })
display(pd.DataFrame(rows).set_index("metric"))


In [ ]:
# ── Visual comparison — spectrogram grid ──────────────────────────────────
# TODO: implement once ThePainter is ready.
# For N_VIS samples from val_loader:
#   Row 1: real rendered mel
#   Row 2: production predicted mel
#   Row 3: candidate predicted mel
# Use matplotlib imshow with shared colour scale.
print("Spectrogram grid: TODO")


In [ ]:
# ── Promote ───────────────────────────────────────────────────────────────
prod_mel_l1 = prod_metrics.get("mel_l1", float("inf"))
cand_mel_l1 = cand_metrics.get("mel_l1", float("inf"))
if cand_mel_l1 < prod_mel_l1:
    print(f"Candidate improves mel_l1: {prod_mel_l1:.4f} → {cand_mel_l1:.4f}")
else:
    print(f"Candidate does NOT improve mel_l1 ({cand_mel_l1:.4f} vs {prod_mel_l1:.4f}). Promote anyway?")

show_register_widget(RUN_ID, "model", model_name=MODEL_NAME)
show_describe_widget(RUN_ID, MLFLOW_URI)
